# Superstore — Notebook nâng cấp: Kiểm tra dữ liệu và phân tích lợi nhuận

**Mục tiêu:** đọc CSV gốc, kiểm tra dữ liệu đầu vào, giữ chính xác Sales/Profit, phân tích KPI, ngành hàng và mức giảm giá.

**Cách chạy:** đặt `Sample - Superstore.csv` cạnh notebook (hoặc đổi biến `INPUT_FILE` tới file dữ liệu tải lên Google Colab). Chạy lần lượt các cell. Công thức không khẳng định quan hệ .

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

INPUT_FILE = Path('Sample - Superstore.csv')
OUTPUT_FILE = Path('Cleaned_Superstore_v2.csv')
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Tệp nguồn:', INPUT_FILE.resolve())


## 1. Đọc dữ liệu và kiểm tra chất lượng

CSV gốc trên GitHub sử dụng bảng mã `windows-1252`. Không dùng `dropna()` bừa bãi trên mọi cột; hãy đo số lượng NULL và kiểm tra các trường quan trọng trước.


In [ ]:
df = pd.read_csv(INPUT_FILE, encoding='cp1252', dtype={'Postal Code': 'string'})
required = ['Row ID','Order ID','Order Date','Ship Date','Customer ID',
            'Category','Sub-Category','Sales','Profit','Quantity','Discount']
missing_columns = sorted(set(required) - set(df.columns))
if missing_columns:
    raise ValueError(f'Thiếu cột bắt buộc: {missing_columns}')
print('Số dòng gốc:', len(df))
print('Số hàng bị trùng Row ID:', df['Row ID'].duplicated().sum())
print('Giá trị thiếu theo cột (chỉ hiện cột có thiếu):')
print(df.isna().sum().loc[lambda x: x > 0])
print('Số hàng thiếu cột quan trọng:', df[required].isna().any(axis=1).sum())
print('Số lượng <= 0:', (pd.to_numeric(df['Quantity'], errors='coerce') <= 0).sum())
print('Doanh thu âm:', (pd.to_numeric(df['Sales'], errors='coerce') < 0).sum())
print('Discount ngoài khoảng [0,1]:', (~pd.to_numeric(df['Discount'], errors='coerce').between(0,1)).sum())


## 2. Chuẩn hóa kiểu dữ liệu, không làm tròn từng dòng

Ngày trong CSV nguồn ở dạng **tháng/ngày/năm**. Việc làm tròn `Sales`/`Profit` được hoãn tới phần **hiển thị**, để tránh sai lệch tổng doanh thu/lợi nhuận vài cent.


In [ ]:
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y', errors='coerce')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%m/%d/%Y', errors='coerce')
for col in ['Sales', 'Profit', 'Quantity', 'Discount']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
if df[required].isna().any(axis=1).any():
    raise ValueError('Có dữ liệu thiếu hoặc không chuyển đổi được trong các cột quan trọng. Hãy kiểm tra trước khi loại dòng.')
if df['Row ID'].duplicated().any():
    raise ValueError('Row ID bị trùng. Cần xác minh nguồn.')
if (df['Quantity'] <= 0).any() or (df['Sales'] < 0).any() or (~df['Discount'].between(0, 1)).any():
    raise ValueError('Quantity/Sales/Discount có giá trị bất thường. Cần kiểm tra dữ liệu.')
if (df['Ship Date'] < df['Order Date']).any():
    raise ValueError('Phát hiện Ship Date trước Order Date.')

df['Year'] = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month
df['Year-Month'] = df['Order Date'].dt.to_period('M').astype(str)
df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8-sig')
print(f'Đã lưu: {OUTPUT_FILE} | số dòng: {len(df):,}')


## 3. KPI cốt lõi

**Phân biệt dòng giao dịch và số đơn hàng.** `Order ID` lặp nếu một đơn có nhiều sản phẩm; cần `nunique` để tính số đơn.


In [ ]:
kpi = pd.Series({
    'Order Lines': len(df),
    'Unique Orders': df['Order ID'].nunique(),
    'Unique Customers': df['Customer ID'].nunique(),
    'Quantity': df['Quantity'].sum(),
    'Total Sales': df['Sales'].sum(),
    'Total Profit': df['Profit'].sum(),
    'Profit Margin (%)': 100 * df['Profit'].sum() / df['Sales'].sum(),
    'Loss-making Lines': (df['Profit'] < 0).sum(),
})
print(kpi.to_string())


## 4. Phân tích theo năm, danh mục và Sub-Category

`Profit Margin = SUM(Profit)/SUM(Sales)` ở từng nhóm, **không** lấy trung bình Profit Margin của từng dòng.


In [ ]:
def group_sales_profit(data, keys):
    output = data.groupby(keys, dropna=False, observed=False).agg(
        Lines=('Row ID', 'size'), Sales=('Sales', 'sum'),
        Profit=('Profit', 'sum'), Quantity=('Quantity', 'sum')
    ).reset_index()
    output['Profit Margin (%)'] = np.where(
        output['Sales'].ne(0), 100 * output['Profit'] / output['Sales'], np.nan
    )
    return output

print('Theo năm:')
display(group_sales_profit(df, ['Year']).sort_values('Year'))
print('Theo danh mục:')
display(group_sales_profit(df, ['Category']).sort_values('Sales', ascending=False))
print('Sub-Category có lợi nhuận thấp nhất:')
display(group_sales_profit(df, ['Sub-Category']).sort_values('Profit').head(8))


## 5. Phân nhóm mức giảm giá và đánh giá mối liên hệ với lợi nhuận

Phân nhóm Discount giúp khoanh vùng sản phẩm cần kiểm tra.


In [ ]:
bins = [-0.001, 0, 0.1, 0.2, 0.3, 0.4, 1.0]
labels = ['0%', '(0%, 10%]', '(10%, 20%]', '(20%, 30%]', '(30%, 40%]', '>40%']
df['Discount Band'] = pd.cut(df['Discount'], bins=bins, labels=labels, include_lowest=True)
by_discount = group_sales_profit(df, ['Discount Band'])
loss_counts = df.assign(is_loss=df['Profit'] < 0).groupby('Discount Band', observed=False)['is_loss'].sum()
by_discount['Loss-making Lines'] = by_discount['Discount Band'].map(loss_counts).astype(int)
display(by_discount)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(by_discount['Discount Band'].astype(str), by_discount['Profit'])
axes[0].set_title('Tổng lợi nhuận theo mức giảm giá')
axes[0].set_ylabel('Profit (USD)')
axes[0].tick_params(axis='x', rotation=35)
axes[1].scatter(df['Discount'], df['Profit'], alpha=0.15, s=9)
axes[1].axhline(0, linewidth=1)
axes[1].set_title('Discount và Profit ở cấp dòng giao dịch')
axes[1].set_xlabel('Discount (0–1)')
axes[1].set_ylabel('Profit (USD)')
plt.tight_layout()
plt.show()


## 6. Phân tích Segment, Region và xu hướng theo tháng

Đề xuất kiểm tra thêm Category/State trước khi diễn giải một khu vực hay phân khúc có hiệu quả thấp.


In [ ]:
print('Theo phân khúc:')
display(group_sales_profit(df, ['Segment']).sort_values('Profit', ascending=False))
print('Theo khu vực:')
display(group_sales_profit(df, ['Region']).sort_values('Profit', ascending=False))
print('Theo năm-tháng:')
monthly = group_sales_profit(df, ['Year-Month']).sort_values('Year-Month')
display(monthly.tail(12))
ax = monthly.plot(x='Year-Month', y=['Sales', 'Profit'], figsize=(14, 4), rot=90)
ax.set_title('Doanh thu và lợi nhuận theo tháng')
ax.set_xlabel('Năm - tháng')
ax.set_ylabel('USD')
plt.tight_layout()
plt.show()


## 7. Kiểm chứng và kết luận

- Chạy notebook bằng CSV nguồn gốc trước khi cập nhật Power BI.
- Kiểm tra dòng trước/sau không mất bất thường.
- Notebook ban đầu **làm tròn từng dòng**, vì vậy số liệu cộng từ `Cleaned_Superstore.csv` có thể lệch nhẹ với bản mới.
- Đối chiếu KPI trên Power BI theo **đúng CSV** hiện được PBIX sử dụng.
- Đề xuất tiếp theo: báo cáo Profit Margin theo `Sub-Category × Discount Band × Region` và đánh giá nguyên nhân bằng dữ liệu bổ sung.
